# 03 — Prétraitement et traits de surface — mesurer le prix du vocabulaire

**Projet** : Routage de tickets de support par TF-IDF — la référence explicable

## Objectifs pédagogiques

1. Suivre une phrase de ticket à travers normalisation, tokenisation et mots vides.
1. Décrire les textes avec des traits de surface, par classe.
1. Mesurer ce que `min_df` retire au vecteur et ce qu'il coûte aux textes inédits.
1. Vérifier qu'aucun trait de surface ne suffit à prédire le libellé.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = [
    "canonique",
    "paraphrase",
    "bruite",
]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

## 2. D'une phrase brute à une suite de tokens

La normalisation est un choix, pas une évidence : elle décide de ce que le modèle pourra confondre. On la lit sur un exemple **bruité**, parce que c'est là qu'elle travaille : politesse, majuscules et référence de commande.

In [ ]:
from src.preprocessing.transformers import (
    TextPreprocessingPipeline,
    normalise_text,
    tokenize,
)

example = DOCUMENTS.loc[DOCUMENTS["style"] == "bruite", "text"].iloc[0]
print("brut      :", str(example)[:180])
print("normalisé :", normalise_text(str(example))[:180])
print("tokens    :", tokenize(example)[:14], f"... ({len(tokenize(example))} tokens)")

preprocessing = TextPreprocessingPipeline.from_config(node(CONFIG, "preprocessing"))
print("politique de découpage :", dict(node(CONFIG, "preprocessing")).get("chunking", {}))
print("tokenisation du pipeline :", preprocessing.transform([str(example)])[0][:14], "...")

stop_words = preprocessing.stop_words
kept_before = tokenize(example)
kept_after = stop_words.transform(kept_before)
print(f"avant filtre : {len(kept_before)} tokens | après filtre : {len(kept_after)}")
print("mots vides retirés :", [token for token in kept_before if token not in kept_after][:10])

**Ce qu'il faut retenir**

- La normalisation met en minuscules et déplie les espaces : « CMD-12345 » devient un token stable, que la référence soit écrite en majuscules ou non.
- Les mots vides sont retirés **par le même code** que celui de l'entraînement : le notebook ne réimplémente rien.
- Une phrase de politesse produit des tokens sans information : ils seront présents dans toutes les classes — donc inutilisables comme signal, mais capables de noyer les termes utiles si le vecteur les sur-pondère.

## 3. Traits de surface, par classe

Avant de vectoriser, on décrit. Ces traits ne servent pas de features au modèle principal : ils servent à **comprendre** ce que le corpus contient, et à vérifier qu'il ne contient pas de raccourci.

In [ ]:
from src.features.build_features import build_text_features, feature_summary

enriched = build_text_features(TRAIN)
surface = feature_summary(enriched)
display(surface.round(3))

In [ ]:
profiles = surface.drop(index="all")
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(profiles.index, profiles["n_words"], color="#4c72b0")
axes[0].set_title("Longueur moyenne par classe (tokens)")
axes[0].set_xlabel("tokens")
axes[1].barh(profiles.index, profiles["has_signature"], color="#55a868")
axes[1].set_title("Part des tickets signés")
axes[1].set_xlabel("part")
figure.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Le tableau par classe est la première lecture d'un corpus de texte : il dit si les classes se ressemblent en surface.
- Les écarts de longueur sont modestes — c'est voulu : la forme ne doit pas être un signal, sinon le projet mesurerait la verbosité des clients.
- La ligne `all` (toute la population) sert de référence : un segment qui s'en écarte fortement est un segment qu'il faudra regarder en particulier.

## 4. Mesurer le prix d'un vocabulaire

`min_df` jette les termes trop rares. Trop petit, le vecteur mémorise des coquilles ; trop grand, un texte qui utilise une tournure inédite n'a plus rien à dire au modèle. On **mesure** les deux effets sur le corpus du projet plutôt que de recopier une valeur par défaut.

In [ ]:
from src.preprocessing.transformers import LexicalVectorizer

lexical_config = dict(dict(node(CONFIG, "model").get("params", {})).get("lexical", {}))
train_texts = TRAIN[TEXT_COLUMN].astype(str).tolist()
test_texts = TEST[TEXT_COLUMN].astype(str).tolist()
test_tokens = [len(tokenize(text)) for text in test_texts]

rows = []
for min_df in (1, 2, 3, 5):
    vectorizer = LexicalVectorizer.from_config({**lexical_config, "min_df": min_df})
    vectorizer.fit(train_texts)
    matrix = np.asarray(vectorizer.transform(test_texts))
    known = np.asarray((matrix != 0).sum(axis=1)).ravel()
    rows.append(
        {
            "min_df": min_df,
            "vocabulaire": vectorizer.vocabulary_size,
            "termes connus par ticket": round(float(known.mean()), 2),
            "tickets muets": int((known == 0).sum()),
            "part des tickets muets": round(float((known == 0).mean()), 4),
        }
    )
grid = pd.DataFrame(rows).set_index("min_df")
display(grid)
print(
    "rappel : le vecteur indexe des unigrammes **et** des bigrammes — "
    "« termes connus par ticket » se lit en variation, pas comme un nombre de mots."
)

**Ce qu'il faut retenir**

- Le vocabulaire fond quand `min_df` monte : les tournures rares disparaissent du vecteur, et la couverture des tokens du test baisse avec elles.
- Un ticket **muet** (aucun terme connu) est le cas extrême : il est parfaitement lisible pour un humain et totalement invisible pour un modèle lexical.
- La valeur de `min_df` retenue par le projet est celle du manifeste : elle est visible, comparable et rediscutable sur ces deux colonnes.

## 5. Le test de raccourci, trait par trait

Même instrument que le notebook 01, mais vu par trait : quel trait unique suffirait à router les tickets ? La réponse doit rester « aucun, tant s'en faut ».

In [ ]:
from src.features.build_features import (
    TEXT_FEATURES,
    group_gain,
    shortcut_scores,
    stump_gain,
)

candidates = [*TEXT_FEATURES, "source", "priority"]
scores = shortcut_scores(DOCUMENTS)


def gain(name: str) -> float:
    """Gain de la meilleure règle à un seuil sur un trait (0,5 : sépare deux classes d'un coup)."""
    if name in TEXT_FEATURES:
        return stump_gain(enriched, name, label_column=TARGET_COLUMN)
    return group_gain(DOCUMENTS, name, label_column=TARGET_COLUMN)


gains = (
    pd.DataFrame({"trait": candidates, "gain": [gain(name) for name in candidates]})
    .set_index("trait")
    .sort_values("gain", ascending=False)
)
gains["marge_vs_plancher"] = (gains["gain"] - scores["majority_baseline"]).round(4)
display(gains.round(4))

best = gains.index[0]
print(
    f"meilleur trait : {best} (gain {gains.loc[best, 'gain']:.4f}) — "
    f"plancher : {scores['majority_baseline']:.4f}"
)

**Ce qu'il faut retenir**

- Un gain de 0,5 signifie qu'un seul seuil sur ce trait sépare parfaitement deux classes : aucun trait du corpus n'en approche.
- Les traits de forme restent sous le plancher majoritaire + 0,05 : la forme ne route pas les tickets.
- Le jour où un trait dépasserait ce niveau, le rapport le signalerait comme raccourci et une revue de modèle serait obligatoire avant toute mise en production.